<a href="https://colab.research.google.com/github/dang-ngan/The-Impact-of-Ban-the-Box-Policies-on-Employment-for-Different-Demographics-in-the-U.S./blob/main/The_Impact_of_%22Ban_the_Box%22_Policies_on_Employment_for_Different_Demographics_in_the_U_S_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Results and Discussion: The Heterogeneous Impact of "Ban the Box" in the US**

"Ban the Box" are regulations demanding employers consider a job candidate’s qualifications first, without the stigma of a conviction or arrest record. Borne out of the work of *All of Us or None*, these policies provide applicants a fair chance at employment by removing conviction and arrest history questions from job applications and delaying background checks until later in the hiring process.

---

***California adopted statewide "Ban the Box" legislation, formally known as the Fair Chance Act (Assembly Bill 1008), in October 2017. The law went into effect on January 1, 2018. Texas has not adopted a statewide "ban the box" law as of early 2025.***

---



### **1. The Average Treatment Effect (ATE)**

Baseline Difference-in-Differences (DiD) model, estimated via Weighted Least Squares (WLS), focuses on the non-college population (Ages 25-54) between 2016 and 2019. The model identifies the overall policy impact using the following specification:  
<br>
 $$Employed_{it} = \alpha + \beta(Treat_i \times Post_t) + \gamma X_{it} + \delta_s + \theta_t + \epsilon_{it}$$
<br>
* Interaction Coefficient ($\beta$): `0.0013`
* Statistical Significance: **p < 0.01**  

Following the 2018 implementation of the California Fair Chance Act, the probability of employment for low-skilled workers in California increased by 0.13 percentage points relative to the control group in Texas. While this suggests that removing the "criminal record box" lowered entry barriers for the general labor force, this marginal average effect is misleading.

---

### **2. Evidence of Statistical Discrimination (Race Interaction)**

By interacting the treatment with racial categories, we find a divergence in outcomes that supports the **Statistical Discrimination** hypothesis (Agan & Starr, 2018):

* Black Applicants: Employment probability **decreased by 8.5%** ($\beta = -0.0851, p < 0.01$).
* White Applicants: Employment probability **increased by 2.2%** ($\beta = 0.0226, p < 0.01$).

When employers are prohibited from asking about criminal history, they lose an objective signal for individual risk. In the absence of data, they may rely on group-level proxies - such as race - to guess the likelihood of a prior conviction. Results suggest that California employers may have reflexively reduced hiring for Black candidates to avoid perceived risk, unintentionally worsening racial disparities.

---

### **3. Labor Market Substitution (Gender Interaction)**

The gender-specific results reveal a "seesaw" effect, indicating a direct competition for roles within the low-education labor market:

* Men: Observed an **11.04% boost** in employment probability.
* Women: Observed an **11.07% decline** in employment probability.

Men constitute roughly 90% of the formerly incarcerated population and are the primary beneficiaries of this policy. Results suggest a **Substitution Effect**: as men with records become more competitive at the screening stage, they appear to be displacing low-skilled women - a group that previously held a relative advantage simply due to lower average incarceration rates.

---

### **4. Model Validity: Parallel Trends**

The validity of these estimates relies on the **Parallel Trends Assumption**. As illustrated in the Event Study (Figure 5), employment trends in California and Texas were statistically indistinguishable from 2016 through 2017.

The sharp divergence occurring exactly in **2018Q1** - coupled with high Z-scores - provides strong empirical evidence that these shifts are a direct consequence of the Fair Chance Act rather than underlying economic drift or state-specific volatility.

---

### **5. Policy Implications and Conclusion**

Findings present a **Policy Paradox**. While "Ban the Box" successfully facilitates labor market entry for the formerly incarcerated (primarily men), it triggers proxy-hiring mechanisms that disadvantage Black applicants and low-skilled women.

**[Final Takeaway]:**
From an econometric standpoint, this study highlights the danger of relying solely on **Average Treatment Effects**. For policymakers, the results suggest that hiding information is not a neutral act; without complementary anti-discrimination enforcement, removing "the box" may simply replace one barrier with another based on demographic stereotypes.

## **Useful References**

1. Agan, A., & Starr, S. (2016). Ban the Box, Criminal Records, and Statistical Discrimination: A Field Experiment. Quarterly Journal of Economics. https://law.yale.edu/sites/default/files/area/workshop/leo/leo16_starr.pdf

2. Doleac, J. L., & Hansen, B. (2016). Does "Ban the Box" Help or Hurt Low-Skilled Workers? Statistical Discrimination and Employment Outcomes when Criminal Histories are Hidden. Journal of Labor Economics. https://www.nber.org/system/files/working_papers/w22469/w22469.pdf
  
3. Shoag, D., & Veuger, S. (2021). Ban-the-Box Measures Help High-Crime Neighborhoods. Journal of Law and Economics. https://www.aei.org/wp-content/uploads/2022/05/Veuger-JLE-Report.pdf?x85095

4. Jackson, O., & Zhao, B. (2017). The Effect of Changing Employers' Access to Criminal Histories on Ex-Offenders' Labor Market Outcomes. Federal Reserve Bank of Boston. https://www.bostonfed.org/-/media/Documents/Workingpapers/PDF/wp1630.pdf

5. Pager, D. (2003). The Mark of a Criminal Record. American Journal of Sociology. https://faculty.washington.edu/matsueda/courses/587/readings/Pager%202003%20Mark.pdf

6. California Civil Rights Department (2018). Fair Chance Act (AB 1008) Implementation Guide. https://calcivilrights.ca.gov/fair-chance-act/

7. Craigie, T. A. (2020). Ban the Box, Convictions, and Public Employment. https://scholarworks.smith.edu/cgi/viewcontent.cgi?article=1068&context=eco_facpubs

8. Rose, E. K. (2021). Does Banning the Box Help Ex-Offenders Get Jobs? https://ekrose.github.io/files/btb_seattle_0418.pdf

9. Burton, A. M., & Wasser, M. (2022). Ban the Box and Cross-Border Spillovers. https://annemburton.com/pages/working_papers/Burton_Wasser_BTB.pdf

In [ ]:
!pip install statsmodels

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np
import os

FILE_PATH = 'data_ban_the_box.csv' #insert your local file path
TREATMENT_STATE = 'california'
CONTROL_STATE = 'texas'
POLICY_YEAR = 2018
POLICY_QUARTER = 1

def export_model_results(model, name):
    res = pd.DataFrame({
        'term': model.params.index,
        'coef': model.params.values,
        'lower': model.conf_int()[0].values,
        'upper': model.conf_int()[1].values
    })
    res.to_csv(f'tableau_results_{name}.csv', index=False)

def prepare_robust_data(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    required_cols = ["year", "month", "statefip", "age", "sex", "race", "educ", "empstat", "wtfinl"]
    df = df[required_cols].copy()
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df = df[df['age'].between(25, 54)]
    df['month_num'] = pd.to_datetime(df['month'], format='%B').dt.month
    df['quarter'] = ((df['month_num'] - 1) // 3) + 1
    df['year_quarter'] = df['year'].astype(str) + "Q" + df['quarter'].astype(str)
    df = df[df['year'].between(2016, 2019)]
    educ_mapping = {
        'none or preschool': 1, 'grades 1, 2, 3, or 4': 10,
        'grades 5 or 6': 20, 'grades 7 or 8': 30, 'grade 9': 40,
        'grade 10': 50, 'grade 11': 60, '12th grade, no diploma': 71,
        'high school diploma or equivalent': 73,
        'some college but no degree': 81,
        "bachelor's degree": 111,
        "master's degree": 123,
        'professional school degree': 124, 'doctorate degree': 125
    }
    df['educ_numeric'] = df['educ'].map(educ_mapping).fillna(0)
    df['education_level'] = pd.cut(
        df['educ_numeric'],
        bins=[-1, 72, 80, 110, np.inf],
        labels=['Less than HS', 'HS Diploma', 'Some College', 'Bachelors+']
    )
    df['employed'] = np.where(df['empstat'].isin(['at work', 'has job, not at work']), 1, 0)
    df['treat'] = np.where(df['statefip'].str.title() == TREATMENT_STATE, 1, 0)
    df['post'] = np.where((df['year'] > POLICY_YEAR) |
                          ((df['year'] == POLICY_YEAR) & (df['quarter'] >= POLICY_QUARTER)), 1, 0)
    df['race_category'] = np.select(
        [df['race'] == 'white', df['race'] == 'black', df['race'] == 'asian only'],
        ['White', 'Black', 'Asian'], default='Other'
    )
    return df.dropna(subset=['wtfinl', 'education_level']).reset_index(drop=True)

def run_robust_did(df):
    formula = """
    employed ~ treat:post + age + I(age**2) + C(sex) +
                C(race_category) + C(education_level) +
                C(statefip) + C(year_quarter)
    """
    model = smf.wls(formula, data=df, weights=df['wtfinl']).fit(
        cov_type='cluster', cov_kwds={'groups': df['statefip']}
    )
    print(model.summary().tables[1])
    export_model_results(model, "full_did")
    return model

def run_robust_event_study(df):
    df = df.copy()
    df['event_time'] = (df['year'] - POLICY_YEAR) * 4 + (df['quarter'] - POLICY_QUARTER)
    df = df[df['event_time'].between(-8, 7)].reset_index(drop=True)

    df.to_csv('tableau_clean_microdata_event_study.csv', index=False)

    formula = """
    employed ~ C(event_time, Treatment(reference=-1)):treat +
                age + C(sex) + C(race_category) + C(education_level) +
                C(statefip) + C(year_quarter)
    """
    model = smf.wls(formula, data=df, weights=df['wtfinl']).fit(
        cov_type='cluster', cov_kwds={'groups': df['statefip']}
    )

    res = pd.DataFrame({
        'coef': model.params.filter(like='C(event_time)'),
        'lower': model.conf_int().filter(like='C(event_time)', axis=0)[0],
        'upper': model.conf_int().filter(like='C(event_time)', axis=0)[1]
    })
    res.index = [int(i.split('[T.')[1].split(']')[0]) for i in res.index]
    res.loc[-1] = [0, 0, 0]
    res = res.sort_index()
    res.to_csv('tableau_event_study_coefficients.csv')
    return model

def run_subgroup_analysis_filtered(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    required_cols = ["year", "month", "statefip", "age", "sex", "race", "educ", "empstat", "wtfinl"]
    df = df[required_cols].copy()
    df['statefip'] = df['statefip'].str.lower()
    df = df[df['statefip'].isin([TREATMENT_STATE, CONTROL_STATE])]
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df = df[df['age'].between(25, 54)]
    df['month_num'] = pd.to_datetime(df['month'], format='%B').dt.month
    df['quarter'] = ((df['month_num'] - 1) // 3) + 1
    df['year_quarter'] = df['year'].astype(str) + "Q" + df['quarter'].astype(str)
    df = df[df['year'].between(2016, 2019)].copy()
    df['employed'] = np.where(df['empstat'].isin(['at work', 'has job, not at work']), 1, 0)
    df['treat'] = np.where(df['statefip'] == TREATMENT_STATE, 1, 0)
    df['post'] = np.where((df['year'] > POLICY_YEAR) |
                          ((df['year'] == POLICY_YEAR) & (df['quarter'] >= POLICY_QUARTER)), 1, 0)
    low_educ_categories = [
        'high school diploma or equivalent', '12th grade, no diploma',
        'grade 11', 'grade 10', 'grade 9', 'grades 7 or 8',
        'grades 5 or 6', 'grades 1, 2, 3, or 4', 'none or preschool'
    ]
    df_subgroup = df[df['educ'].str.lower().isin(low_educ_categories)].copy()
    df_subgroup['race_category'] = np.select(
        [df_subgroup['race'] == 'white', df_subgroup['race'] == 'black'],
        ['White', 'Black'], default='Other'
    )

    df_subgroup.to_csv('tableau_subgroup_non_college.csv', index=False)

    formula = """
    employed ~ treat:post + age + I(age**2) + C(sex) +
                C(race_category) + C(statefip) + C(year_quarter)
    """
    model = smf.wls(formula, data=df_subgroup, weights=df_subgroup['wtfinl']).fit(
        cov_type='cluster', cov_kwds={'groups': df_subgroup['statefip']}
    )
    export_model_results(model, "subgroup_did")
    return model

def test_racial_impact(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    df['statefip'] = df['statefip'].str.lower()
    df = df[df['statefip'].isin(['california', 'texas'])]
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df = df[df['age'].between(25, 54) & df['year'].between(2016, 2019)]
    df['month_num'] = pd.to_datetime(df['month'], format='%B').dt.month
    df['quarter'] = ((df['month_num'] - 1) // 3) + 1
    df['year_quarter'] = df['year'].astype(str) + "Q" + df['quarter'].astype(str)
    df['employed'] = np.where(df['empstat'].isin(['at work', 'has job, not at work']), 1, 0)
    df['treat'] = np.where(df['statefip'] == 'california', 1, 0)
    df['post'] = np.where((df['year'] > 2018) | ((df['year'] == 2018) & (df['quarter'] >= 1)), 1, 0)
    low_educ = ['high school diploma or equivalent', '12th grade, no diploma', 'grade 11', 'grade 10', 'grade 9']
    df_sub = df[df['educ'].str.lower().isin(low_educ)].copy()
    df_sub['race_category'] = np.select([df_sub['race'] == 'white', df_sub['race'] == 'black'], ['White', 'Black'], default='Other')

    formula = """
    employed ~ treat:post:C(race_category) + age + I(age**2) + C(sex) +
                C(race_category) + C(statefip) + C(year_quarter)
    """
    model = smf.wls(formula, data=df_sub, weights=df_sub['wtfinl']).fit(
        cov_type='cluster', cov_kwds={'groups': df_sub['statefip']}
    )
    export_model_results(model, "racial_impact")
    return model

def test_gender_impact(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    df['statefip'] = df['statefip'].str.lower()
    df = df[df['statefip'].isin(['california', 'texas'])]
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    df = df[df['age'].between(25, 54) & df['year'].between(2016, 2019)]
    df['month_num'] = pd.to_datetime(df['month'], format='%B').dt.month
    df['quarter'] = ((df['month_num'] - 1) // 3) + 1
    df['year_quarter'] = df['year'].astype(str) + "Q" + df['quarter'].astype(str)
    df['employed'] = np.where(df['empstat'].isin(['at work', 'has job, not at work']), 1, 0)
    df['treat'] = np.where(df['statefip'] == 'california', 1, 0)
    df['post'] = np.where((df['year'] > 2018) | ((df['year'] == 2018) & (df['quarter'] >= 1)), 1, 0)
    low_educ = ['high school diploma or equivalent', '12th grade, no diploma', 'grade 11', 'grade 10', 'grade 9']
    df_sub = df[df['educ'].str.lower().isin(low_educ)].copy()

    formula = """
    employed ~ treat:post:C(sex) + age + I(age**2) +
                C(race) + C(statefip) + C(year_quarter)
    """
    model = smf.wls(formula, data=df_sub, weights=df_sub['wtfinl']).fit(
        cov_type='cluster', cov_kwds={'groups': df_sub['statefip']}
    )
    export_model_results(model, "gender_impact")
    return model

if __name__ == "__main__":
    full_df = prepare_robust_data(FILE_PATH)
    run_robust_did(full_df)
    run_robust_event_study(full_df)

    run_subgroup_analysis_filtered(FILE_PATH)
    test_racial_impact(FILE_PATH)
    test_gender_impact(FILE_PATH)

In [ ]:
race_labels = ['Black', 'Other', 'White']
race_coefs = [-0.0851, -0.0268, 0.0226]
race_err = [0.018, 0.003, 0.001]

gender_labels = ['Female', 'Male']
gender_coefs = [-0.1107, 0.1104]
gender_err = [0.001, 0.00000935]

inter_labels = ['Black Men', 'White Men', 'Black Women', 'White Women']
inter_coefs = [-0.062, 0.132, -0.095, -0.015]

plt.rcParams.update({'font.size': 12, 'axes.labelweight': 'bold'})

plt.figure(figsize=(10, 6))
plt.bar(race_labels, race_coefs, yerr=[1.96*e for e in race_err], capsize=8,
        color=['#d9534f', '#f0ad4e', '#5cb85c'], alpha=0.8, edgecolor='black')
plt.axhline(0, color='black', linewidth=1.5)
plt.title('Figure 1: Racial Heterogeneity in "Ban the Box" Effects', fontsize=15)
plt.ylabel('Change in Employment Probability')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.bar(gender_labels, gender_coefs, yerr=[1.96*e for e in gender_err], capsize=10,
        color=['#d9534f', '#5cb85c'], alpha=0.8, edgecolor='black')
plt.axhline(0, color='black', linewidth=1.5)
plt.title('Figure 2: Gender-Based Job Substitution', fontsize=15)
plt.ylabel('Change in Employment Probability')
plt.show()

plt.figure(figsize=(10, 7))
all_labels = ['Black', 'Other', 'White', 'Male', 'Female']
all_coefs = race_coefs + gender_coefs
all_err = race_err + gender_err

plt.errorbar(all_coefs, range(len(all_labels)), xerr=[1.96*e for e in all_err],
             fmt='o', color='navy', markersize=10, capsize=6, linewidth=2)
plt.yticks(range(len(all_labels)), all_labels)
plt.axvline(0, color='red', linestyle='--', alpha=0.6)
plt.title('Figure 3: Summary of Treatment Effects Across All Subgroups', fontsize=15)
plt.xlabel('Policy Effect Estimate')
plt.gca().invert_yaxis()
plt.grid(alpha=0.2)
plt.show()

plt.figure(figsize=(10, 6))
plt.bar(inter_labels, inter_coefs, color=['#c0392b', '#27ae60', '#e74c3c', '#f1c40f'], alpha=0.8)
plt.axhline(0, color='black', linewidth=1)
plt.title('Figure 4: Intersectional Impact (Combined Effects)', fontsize=15)
plt.ylabel('Net Employment Change')
plt.xticks(rotation=15)
plt.show()